# Week 8 — Tuesday Lab Notebook  
## Data Preprocessing with scikit-learn

### Today’s Goal

By the end of today, students should be able to:

- understand why preprocessing is needed before model training
- separate **numeric** and **categorical** columns
- handle missing values in a proper way
- encode categorical variables using **one-hot encoding**
- scale numeric data using **StandardScaler**
- split data into train and test correctly
- understand what a **Pipeline** does
- understand what a **ColumnTransformer** does
- build one clean preprocessing + model workflow

## 3-Hour Structure

**0:00–0:20** Why preprocessing is important  
**0:20–0:45** Build and inspect a messy dataset  
**0:45–1:15** Missing values and imputers  
**1:15–1:40** Encoding categorical data  
**1:40–2:05** Scaling numeric data  
**2:05–2:35** `ColumnTransformer` in simple wording  
**2:35–2:55** `Pipeline` for full workflow  
**2:55–3:00** Practice + wrap-up

## Part A — What is Data Preprocessing?

Data preprocessing means:

> preparing raw data before giving it to a machine learning model.

Real data is often not clean. It may contain:

- missing values
- text labels like `Male`, `Female`, `Yes`, `No`
- columns with very different scales
- inconsistent formats
- unwanted spaces or messy entries

If we do not prepare the data well, the model may:

- give weak results
- fail completely
- learn misleading patterns

## Part B — Why Preprocessing Matters

Suppose we want to predict whether a customer will buy a product.

Our data may have:

- `age`
- `salary`
- `city`
- `gender`
- `experience_years`

Problems:

- `salary` may be missing in some rows
- `city` is text, but many ML models need numbers
- `salary` may be in thousands while `age` is only 20 to 60
- some rows may have blank values

So before training, we need to **clean and transform** the data.

## Part C — Import Libraries

We will use:

- `pandas` and `numpy` for data
- `train_test_split` for splitting
- `SimpleImputer` for missing values
- `OneHotEncoder` for text categories
- `StandardScaler` for scaling
- `ColumnTransformer` for different treatment of different column types
- `Pipeline` for chaining steps together
- `LogisticRegression` as a simple model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Part D — Create a Small Messy Dataset

We will create a customer dataset.

Goal:

- predict whether a customer buys a product or not

Target column:

- `buy_product`

Features:

- `age`
- `salary`
- `experience_years`
- `city`
- `education`

In [ ]:
customers = pd.DataFrame({
    "age": [22, 25, 28, 35, 40, np.nan, 31, 29, 45, 38, 27, 50, 33, np.nan, 41],
    "salary": [30000, 35000, 40000, 60000, 65000, 52000, np.nan, 45000, 80000, 72000, 39000, 90000, 58000, 61000, np.nan],
    "experience_years": [1, 2, 3, 8, 10, 6, 4, 3, 15, 11, 2, 20, 7, 8, 12],
    "city": ["Lahore", "Karachi", "Lahore", "Islamabad", "Karachi", "Lahore", np.nan, "Islamabad", "Karachi", "Lahore", "Karachi", "Islamabad", "Lahore", "Karachi", "Islamabad"],
    "education": ["BS", "BS", "MS", "MS", "PhD", "BS", "BS", np.nan, "PhD", "MS", "BS", "PhD", "MS", "BS", "MS"],
    "buy_product": ["No", "No", "Yes", "Yes", "Yes", "Yes", "No", "No", "Yes", "Yes", "No", "Yes", "Yes", "No", "Yes"]
})

customers

## Part E — First Look at the Data

Before preprocessing, always inspect the data.

In [ ]:
print(customers.head())
print()
print(customers.info())

In [ ]:
customers.describe(include="all")

### What should students notice?

- some values are missing in `age`
- some values are missing in `salary`
- some values are missing in `city`
- some values are missing in `education`
- `city` and `education` are text columns
- target `buy_product` is also text

## Part F — Separate Features and Target

This is the same ML habit from Monday.

- `X` means input features
- `y` means target

In [ ]:
X = customers.drop("buy_product", axis=1)
y = customers["buy_product"]

print("Features:")
print(X.head())
print()
print("Target:")
print(y.head())

## Part G — Split Before Fitting Preprocessing Steps

This is very important.

We should split data into train and test **before** learning things like:

- mean values for imputation
- scaling parameters
- category mappings

Why?

Because if we use all data first, the test set information leaks into training.

That is a form of **data leakage**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Part H — Identify Numeric and Categorical Columns

We often preprocess different column types differently.

### Numeric columns
These are number-based columns.

### Categorical columns
These are text or category columns.

In [ ]:
numeric_features = ["age", "salary", "experience_years"]
categorical_features = ["city", "education"]

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

## Part I — Missing Values in Simple Words

A missing value means some data is not available.

In pandas, missing values often appear as:

- `NaN`
- `None`

Common ways to handle missing values:

1. remove the row  
2. remove the column  
3. fill the missing value with some replacement  

In machine learning, a common replacement is:

- mean for numeric columns
- most frequent category for text columns

In [ ]:
print("Missing values in training data:")
print(X_train.isnull().sum())

## Part J — Numeric Imputation with `SimpleImputer`

For numeric columns, a common choice is:

- `strategy="mean"`

This means:

> replace missing numeric values with the average of the training column

In [ ]:
numeric_imputer = SimpleImputer(strategy="mean")

numeric_imputer.fit(X_train[numeric_features])

X_train_num_imputed = numeric_imputer.transform(X_train[numeric_features])
X_test_num_imputed = numeric_imputer.transform(X_test[numeric_features])

print("Numeric imputed train data:")
print(X_train_num_imputed[:5])

### Important idea

We used:

- `fit()` on training data only
- `transform()` on both training and test data

This is the correct habit.

## Part K — What Did the Imputer Learn?

The imputer stores the fill values it learned from training data.

In [ ]:
print("Means learned from training data:")
for column, value in zip(numeric_features, numeric_imputer.statistics_):
    print(column, "->", value)

## Part L — Categorical Imputation

For text columns, a common choice is:

- `strategy="most_frequent"`

This means:

> replace missing category with the most common category from training data

In [ ]:
categorical_imputer = SimpleImputer(strategy="most_frequent")

categorical_imputer.fit(X_train[categorical_features])

X_train_cat_imputed = categorical_imputer.transform(X_train[categorical_features])
X_test_cat_imputed = categorical_imputer.transform(X_test[categorical_features])

print("Categorical imputed train data:")
print(X_train_cat_imputed[:5])

## Part M — What Did the Categorical Imputer Learn?

In [ ]:
print("Most frequent values learned from training data:")
for column, value in zip(categorical_features, categorical_imputer.statistics_):
    print(column, "->", value)

## Part N — Why Encoding is Needed

Most ML models cannot directly use text such as:

- Lahore
- Karachi
- BS
- MS
- PhD

So we convert categories into numbers.

A beginner may think:

- Lahore = 1
- Karachi = 2
- Islamabad = 3

But that may create a false order.

So a safer method is usually:

## One-Hot Encoding

## Part O — One-Hot Encoding in Simple Words

One-hot encoding creates a new binary column for each category.

Example:

If `city` has:

- Lahore
- Karachi
- Islamabad

Then it becomes:

- `city_Lahore`
- `city_Karachi`
- `city_Islamabad`

Each row gets 0 or 1.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoder.fit(X_train_cat_imputed)

X_train_cat_encoded = encoder.transform(X_train_cat_imputed)
X_test_cat_encoded = encoder.transform(X_test_cat_imputed)

print("Encoded categorical data shape:", X_train_cat_encoded.shape)
print()
print(X_train_cat_encoded[:5])

## Part P — Encoded Column Names

In [ ]:
encoded_feature_names = encoder.get_feature_names_out(categorical_features)
print(encoded_feature_names)

### Why `handle_unknown="ignore"`?

Sometimes in test data, a new category may appear.

With `handle_unknown="ignore"`:

- the code does not crash
- unseen categories are handled safely

## Part Q — Why Scaling is Needed

Some models are affected by the scale of the input.

Example:

- `age` may be around 20 to 50
- `salary` may be around 30000 to 90000

If scales are very different, some models may behave poorly.

A common solution is:

## StandardScaler

## Part R — StandardScaler in Simple Words

`StandardScaler` tries to make each numeric feature have:

- mean close to 0
- standard deviation close to 1

This is called **standardization**.

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train_num_imputed)

X_train_num_scaled = scaler.transform(X_train_num_imputed)
X_test_num_scaled = scaler.transform(X_test_num_imputed)

print("Scaled numeric train data:")
print(np.round(X_train_num_scaled[:5], 2))

## Part S — Check Means After Scaling

In [ ]:
print("Approximate means after scaling:")
print(np.round(X_train_num_scaled.mean(axis=0), 4))

print()
print("Approximate standard deviations after scaling:")
print(np.round(X_train_num_scaled.std(axis=0), 4))

### Important note

Not every model needs scaling equally.

Scaling is especially useful for many models such as:

- logistic regression
- kNN
- SVM
- neural networks

Tree-based models usually do not depend on scaling as much.

## Part T — Manual Combination of Numeric and Categorical Output

Now we have:

- scaled numeric features
- encoded categorical features

We can combine them into one final feature matrix.

In [ ]:
X_train_preprocessed_manual = np.hstack([X_train_num_scaled, X_train_cat_encoded])
X_test_preprocessed_manual = np.hstack([X_test_num_scaled, X_test_cat_encoded])

print("Manual preprocessed train shape:", X_train_preprocessed_manual.shape)
print("Manual preprocessed test shape:", X_test_preprocessed_manual.shape)

## Part U — Why Manual Preprocessing Becomes Messy

Manual preprocessing works for learning.

But in real projects, it becomes difficult because:

- many steps are involved
- easy to forget one step
- training and test steps may get mixed up
- code becomes harder to manage

That is why scikit-learn gives us:

- `ColumnTransformer`
- `Pipeline`

## Part V — `ColumnTransformer` in Simple Words

`ColumnTransformer` allows us to say:

- apply one set of steps to numeric columns
- apply another set of steps to categorical columns

This is very useful when a dataset has mixed column types.

## Part W — Build Numeric and Categorical Pipelines

For numeric columns we will do:

1. impute missing values with mean  
2. scale values  

For categorical columns we will do:

1. impute missing values with most frequent  
2. one-hot encode

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

## Part X — Create the `ColumnTransformer`

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

## Part Y — Fit and Transform with `ColumnTransformer`

This single object now handles:

- missing numeric values
- scaling numeric values
- missing categorical values
- one-hot encoding categorical values

In [ ]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print("Preprocessed train shape:", X_train_preprocessed.shape)
print("Preprocessed test shape:", X_test_preprocessed.shape)

## Part Z — See the Final Feature Names

This is helpful for understanding what happened after preprocessing.

In [ ]:
feature_names_after_preprocessing = preprocessor.get_feature_names_out()
print(feature_names_after_preprocessing)

## Part AA — `Pipeline` in Simple Words

A `Pipeline` connects steps in order.

Example:

1. preprocess data  
2. train model  

This is useful because:

- code is cleaner
- workflow is safer
- train and test handling becomes easier
- cross-validation later becomes easier too

## Part AB — Build a Full Pipeline

We will create:

- preprocessing step
- logistic regression model

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model_pipeline

## Part AC — Train the Full Pipeline

Now we can train in one line.

In [ ]:
model_pipeline.fit(X_train, y_train)

## Part AD — Make Predictions

In [ ]:
y_pred = model_pipeline.predict(X_test)

print("Predictions:")
print(y_pred)
print()
print("Actual values:")
print(y_test.values)

## Part AE — Evaluate the Model

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

### What should students understand here?

Even if our score is not perfect, the main lesson today is:

- how to preprocess properly
- how to build a safe sklearn workflow
- how to avoid messy manual handling

## Part AF — Predict for New Data

We can now use the same pipeline on new raw data.

Notice:

- we do not manually encode or scale again
- pipeline handles all preprocessing automatically

In [ ]:
new_customers = pd.DataFrame({
    "age": [30, 47],
    "salary": [50000, np.nan],
    "experience_years": [5, 14],
    "city": ["Lahore", "Karachi"],
    "education": ["MS", "PhD"]
})

new_predictions = model_pipeline.predict(new_customers)

result = new_customers.copy()
result["predicted_buy_product"] = new_predictions
result

## Part AG — Very Important Beginner Rule

When using sklearn preprocessing:

- `fit` learns from training data
- `transform` applies the learned logic
- `fit_transform` does both together

Usual pattern:

- on training data: `fit_transform`
- on test data: `transform`

## Part AH — Common Beginner Mistakes

1. fitting imputer on full data before split  
2. fitting scaler on full data before split  
3. encoding categories manually in a wrong way  
4. using different preprocessing for train and test  
5. forgetting which columns are numeric and which are categorical  
6. scaling the target by mistake  
7. dropping too much data instead of handling missing values wisely  
8. doing preprocessing in many scattered cells and getting confused

## Part AI — Small Real-Life Example 1: Student Performance

Suppose we want to predict pass/fail.

Possible columns:

- study hours
- attendance
- assignment status
- previous grade
- city
- department

Preprocessing may be needed because:

- some attendance values are missing
- department is text
- study hours and marks may be on different scales

## Part AJ — Small Real-Life Example 2: Employee Attrition

Goal:

- predict whether an employee may leave the company

Possible features:

- age
- salary
- department
- years at company
- education level

Again we may need:

- imputation
- encoding
- scaling

## Part AK — Small Real-Life Example 3: Loan Approval

Goal:

- predict whether loan should be approved

Possible features:

- income
- age
- city
- education
- credit history

This is a very common place where preprocessing is necessary before training.

## Part AL — Mini In-Class Practice

Try these before moving to the after-lab tasks:

1. create `X` and `y` from the customer dataset  
2. split into training and test sets  
3. identify numeric and categorical columns  
4. apply `SimpleImputer` on numeric columns  
5. apply `SimpleImputer` on categorical columns  
6. apply `OneHotEncoder` on categorical data  
7. apply `StandardScaler` on numeric data  
8. combine both using `ColumnTransformer`  
9. build a full `Pipeline` with logistic regression  
10. predict on test data and show accuracy

## Part AM — Recap

Today we learned how to:

- understand preprocessing in simple words
- split data before fitting preprocessing steps
- handle missing values using `SimpleImputer`
- encode text categories using `OneHotEncoder`
- scale numeric features using `StandardScaler`
- use `ColumnTransformer` for mixed column types
- use `Pipeline` for a complete clean workflow
- train and evaluate one end-to-end model

# After-Lab Tasks (10)

### Task 1
Create a small DataFrame with at least 12 rows and these columns: `age`, `income`, `city`, `education`, and `approved`.

### Task 2
From your DataFrame, separate the **features** and the **target**.

### Task 3
Split your data into training and test sets using `train_test_split()`.

### Task 4
Write which columns are **numeric** and which are **categorical**.

### Task 5
Apply `SimpleImputer(strategy="mean")` on your numeric columns.

### Task 6
Apply `SimpleImputer(strategy="most_frequent")` on your categorical columns.

### Task 7
Apply `OneHotEncoder()` to the categorical columns and print the new encoded column names.

### Task 8
Apply `StandardScaler()` to the numeric columns and print the scaled values of the first 5 rows.

### Task 9
Use `ColumnTransformer` to combine numeric and categorical preprocessing in one object.

### Task 10
Create a full `Pipeline` with preprocessing + `LogisticRegression`, train it, predict on test data, and print the accuracy.

# Optional Homework Challenge

Create your own small real-life dataset for one of these problems:

- student pass prediction
- loan approval
- employee attrition
- customer purchase prediction

Then do all of the following:

1. explain the problem in 2 to 3 lines  
2. create at least 15 rows of data  
3. identify numeric and categorical columns  
4. add at least 2 missing values intentionally  
5. build a preprocessing workflow using:
   - `SimpleImputer`
   - `OneHotEncoder`
   - `StandardScaler`
   - `ColumnTransformer`
6. build a full `Pipeline` with a classification model  
7. train and test the model  
8. print accuracy  
9. write 4 to 5 lines explaining what preprocessing steps were used and why